In [0]:
from pyspark.sql.functions import coalesce, try_to_date, col, trim, upper
from pyspark.sql.types import IntegerType



In [0]:
df_sales = spark.table("retailer.bronze.sales_raw")

display(df_sales)

In [0]:
df_sales_clean = (
    df_sales
    .withColumn(
        "order_date",
        coalesce(
            try_to_date(col("order_date"), "dd-MM-yyyy"),
            try_to_date(col("order_date"), "M/d/yyyy")
        )
    )
    .withColumn(
        "delivery_date",
        coalesce(
            try_to_date(col("delivery_date"), "M/d/yyyy"),
            try_to_date(col("delivery_date"), "dd-MM-yyyy")
        )
    )
)

In [0]:
df_sales_clean = df_sales_clean.withColumn(
    "quantity",
    col("quantity").cast(IntegerType())
)

In [0]:
df_sales_clean = df_sales_clean.withColumn(
    "currency_code",
    trim(upper(col("currency_code")))
)

In [0]:
df_sales_clean = (
    df_sales_clean
    .filter(
        col("order_number").isNotNull() &
        col("line_item").isNotNull() &
        col("customer_key").isNotNull() &
        col("store_key").isNotNull()
    )
    .dropDuplicates(["order_number", "line_item"])
)

In [0]:
display(df_sales_clean)

df_sales_clean.printSchema()

In [0]:
df_sales_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.silver.sales")